# CS456 Final Project — Binary Search (Parts iii & iv)
**Owner: Donna (Member D)**

This notebook implements:
1. **The packing check** (`feasible`) — can `k` bags with weight limit `X` hold all the boxes, packed in order?
2. **Binary search** (`binary_search_partition`) — find the smallest limit `X` that passes the check.
3. **Recovering the actual groups** — pack once more with the final limit.
4. **The extension** — at most `L` boxes per group.

Running example used throughout the team guide: `[7, 2, 5, 10, 8]` — answer **18** for `k = 2`, **14** for `k = 3`.

## 1. The packing check (Part iii)

Walk along the boxes in order. Keep filling the current bag; when the next box would push it over the limit `X`, close the bag and open a new one. Pass if we needed at most `k` bags.

Fewer than `k` bags is fine — a bag can always be split later (the exactly-k fix, section 6 of the guide).

In [ ]:
def feasible(weights, k, X):
    """Return True if the boxes can be packed, in order, into at most k bags
    of capacity X. Assumes X >= max(weights)."""
    count = 1          # bags used so far (we always open the first bag)
    current = 0        # weight in the current bag
    for w in weights:
        if current + w > X:      # box doesn't fit: close bag, start a new one
            count += 1
            current = w
        else:
            current += w
    return count <= k

In [ ]:
# The two worked examples from section 4 of the guide
boxes = [7, 2, 5, 10, 8]

print("Limit 18, k=2:", feasible(boxes, 2, 18))   # expected: True  (2 bags)
print("Limit 15, k=2:", feasible(boxes, 2, 15))   # expected: False (needs 3 bags)

Limit 18, k=2: True
Limit 15, k=2: False


## 2. Binary search on the answer (Part iii)

The answer lies between `max(weights)` (no bag can be smaller than the biggest box) and `sum(weights)` (one bag holds everything).

Each round, guess the middle of the range and run the packing check:
- **Pass** → the answer is this guess or smaller → keep the lower half (`hi = mid`).
- **Fail** → the answer is bigger → keep the upper half (`lo = mid + 1`).

This is only valid because feasibility is **monotone** (Part iv): if limit `X` works, every limit `X' ≥ X` works too — the same split still fits in bigger bags. So passes never turn back into fails, and throwing away half the range each round is safe.

In [ ]:
def binary_search_partition(weights, k, verbose=False):
    """Return the minimum possible largest batch sum when splitting
    `weights` (in order) into k consecutive groups. O(n log S) time."""
    lo, hi = max(weights), sum(weights)
    rnd = 0
    while lo < hi:
        mid = (lo + hi) // 2
        ok = feasible(weights, k, mid)
        rnd += 1
        if verbose:
            print(f"Round {rnd}: range [{lo}, {hi}], guess {mid} -> {'Pass' if ok else 'Fail'}")
        if ok:
            hi = mid           # answer is mid or smaller
        else:
            lo = mid + 1       # answer is strictly bigger
    return lo

In [ ]:
# Reproduce the guessing-game table from section 4 (k = 2, answer 18)
answer = binary_search_partition(boxes, 2, verbose=True)
print("\nAnswer for k=2:", answer)      # expected: 18
print("Answer for k=3:", binary_search_partition(boxes, 3))  # expected: 14

Round 1: range [10, 32], guess 21 -> Pass
Round 2: range [10, 21], guess 15 -> Fail
Round 3: range [16, 21], guess 18 -> Pass
Round 4: range [16, 18], guess 17 -> Fail

Answer for k=2: 18
Answer for k=3: 14


## 3. Recovering the actual groups

Binary search only finds the *number*. To get the groups, pack once more using that limit. This may give fewer than `k` groups (e.g. `[1,1,1,10]`, `k=3`, limit 10 packs into 2 bags) — the exactly-k fix (Member A's part, section 6) splits a multi-box group afterwards; packing can also close a bag early when the boxes left equal the bags still needed, which is the O(n) version from Appendix E.

In [ ]:
def recover_groups(weights, k, X):
    """Pack in order with limit X, closing a bag early when the boxes left
    are only just enough for the bags still needed, so we return exactly k groups."""
    n = len(weights)
    groups, current, cur_sum = [], [], 0
    for i, w in enumerate(weights):
        boxes_left = n - i               # boxes not yet placed, including this one
        bags_left = k - len(groups)      # bags not yet closed, including the open one
        # Close the open bag if this box won't fit, OR if adding it would leave
        # too few boxes to give every remaining bag at least one box.
        if current and (cur_sum + w > X or boxes_left < bags_left):
            groups.append(current)
            current, cur_sum = [], 0
        current.append(w)
        cur_sum += w
    groups.append(current)
    return groups

def solve(weights, k):
    """Full solution: the optimal value and one optimal split into exactly k groups."""
    best = binary_search_partition(weights, k)
    return best, recover_groups(weights, k, best)

In [ ]:
print("k=2:", solve(boxes, 2))                 # 18, e.g. [7,2,5] | [10,8]
print("k=3:", solve(boxes, 3))                 # 14, e.g. [7,2,5] | [10] | [8]
print("[1,1,1,10], k=3:", solve([1,1,1,10], 3))  # 10, exactly 3 groups

k=2: (18, [[7, 2, 5], [10, 8]])
k=3: (14, [[7, 2, 5], [10], [8]])
[1,1,1,10], k=3: (10, [[1, 1], [1], [10]])


## 4. Extension: at most L boxes per group (Part 7, binary-search side)

Two changes:
- **Impossible case**: if `k × L < n`, the bags can't hold all the boxes — report no split exists.
- **Packing check**: also close a bag once it already holds `L` boxes.

Monotonicity still holds (bigger bags still work with the same split), so binary search is still valid.

In [ ]:
def feasible_L(weights, k, X, L):
    """Packing check with weight limit X AND at most L boxes per bag."""
    count, current, size = 1, 0, 0
    for w in weights:
        if current + w > X or size == L:   # over weight OR bag already full of boxes
            count += 1
            current, size = w, 1
        else:
            current += w
            size += 1
    return count <= k

def binary_search_partition_L(weights, k, L):
    """Minimum largest batch sum with at most L boxes per group.
    Returns None if no valid split exists (k * L < n)."""
    if k * L < len(weights):
        return None
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible_L(weights, k, mid, L):
            hi = mid
        else:
            lo = mid + 1
    return lo

In [ ]:
# Examples from section 7 of the guide
print("k=3, L=2:", binary_search_partition_L(boxes, 3, 2))  # expected: 15  ([7,2] | [5,10] | [8])
print("k=2, L=2:", binary_search_partition_L(boxes, 2, 2))  # expected: None (2*2 < 5 boxes)

k=3, L=2: 15
k=2, L=2: None


## 5. Sanity tests

Edge cases from Appendix F, plus a brute-force cross-check on small random inputs (brute force is the team's answer key — Member B owns the real one; this mini version is only for checking this notebook).

In [ ]:
from itertools import combinations
import random

def brute_force(weights, k):
    """Try every placement of the k-1 cuts; return the best largest-group sum."""
    n = len(weights)
    prefix = [0]
    for w in weights:
        prefix.append(prefix[-1] + w)
    best = float('inf')
    for cuts in combinations(range(1, n), k - 1):
        edges = (0,) + cuts + (n,)
        heaviest = max(prefix[edges[j+1]] - prefix[edges[j]] for j in range(k))
        best = min(best, heaviest)
    return best

# --- Hand examples ---
assert binary_search_partition([7,2,5,10,8], 2) == 18
assert binary_search_partition([7,2,5,10,8], 3) == 14
assert binary_search_partition([1,1,1,10], 3) == 10

# --- Edge cases ---
assert binary_search_partition([5], 1) == 5                      # one task
assert binary_search_partition([3,1,4,1,5], 1) == 14             # k=1: the total
assert binary_search_partition([3,1,4,1,5], 5) == 5              # k=n: biggest task
assert binary_search_partition([2,2,2,2], 2) == 4                # all equal

# --- Random cross-check against brute force ---
random.seed(456)
for trial in range(1000):
    n = random.randint(1, 10)
    k = random.randint(1, n)
    w = [random.randint(1, 50) for _ in range(n)]
    assert binary_search_partition(w, k) == brute_force(w, k), (w, k)

# --- Groups returned are always valid and match the reported answer ---
for trial in range(1000):
    n = random.randint(1, 10)
    k = random.randint(1, n)
    w = [random.randint(1, 50) for _ in range(n)]
    best, groups = solve(w, k)
    flat = [x for g in groups for x in g]
    assert flat == w, (w, k, groups)                 # order kept, every box once
    assert len(groups) == k, (w, k, groups)          # exactly k groups
    assert all(groups), (w, k, groups)               # none empty
    assert max(sum(g) for g in groups) == best       # heaviest equals the answer

print("All tests passed.")

All tests passed.


## Complexity summary

| Piece | Time | Extra space |
|---|---|---|
| `feasible` | O(n) per call | O(1) |
| `binary_search_partition` | O(n log S), S = sum of weights | O(1) |
| `recover_groups` | O(n) | O(n) for the groups |
| Extension versions | same | same |